In [ ]:
import pandas as pd
import pydeck as pdk
import numpy as np
import streamlit as st

In [ ]:
df_1 = pd.read_csv(r'C:\Users\EC712\OneDrive\Documents\school\DSCI_435\data_sheets\TICKETS_WITH_COORDS.csv')
df_1 = df_1.dropna(subset=['FEP_BUILDING_X_COORDINATE', 'FEP_BUILDING_Y_COORDINATE'])

In [ ]:
# %%
df_1 = pd.read_csv(r'C:\Users\EC712\OneDrive\Documents\school\DSCI_435\data_sheets\TICKETS_WITH_COORDS.csv')
df_1 = df_1.dropna(subset=['FEP_BUILDING_X_COORDINATE', 'FEP_BUILDING_Y_COORDINATE'])

# %%
st.set_page_config(page_title="Interactive Map", layout="wide")

st.sidebar.header("Map Settings")
options_0 = st.sidebar.text_input("Key Word Search", value="")
options_1 = st.sidebar.pills("Service Classes", ['Custodial', 'Capital Projects', 'Electrical', 'Carpentry',
       'Plumbing', 'Moving Equipment', 'Building Interior', 'Grounds',
       'Event Support', 'Building Automation Systems', 'Access Control',
       'Mechanical', 'HVAC', 'Facilities', 'Administrative', 'Lifts', 'Building Exterior', 
       'Life Safety', 'Lab Equipment', 'Instrumentation', 'Cores', 'Networking'], selection_mode="multi", default=[])
options_2 = st.sidebar.pills("Task Type", ['Corrective', 'Preventive', 'Material Order', 'Inspection'], selection_mode="multi", default=[])
options_3 = st.sidebar.pills("Task Priority", ['3 - Routine', '0 - Admin', '2 - Urgent', '4 - Customer Requested',
       '1 - Emergency', 'R1 - Emergency', 'R2 - Routine',
       '7 - Estimate', '9 - Preventive Maintenance',
       '6 - Weather Emergency'], selection_mode="multi", default=[])

if options_0:
  df_1 = df_1[df_1['DESCRIPTION'].str.contains(options_0, na=False)]
if options_1:
  df_1 = df_1[df_1['SERVICE_CLASS'].isin(options_1)]
if options_2:
  df_1 = df_1[df_1['TASK_TYPE'].isin(options_2)]
if options_3:
  df_1 = df_1[df_1['TASK_PRIORITY'].isin(options_3)]


df_filter = df_1.groupby(['FEP_BUILDING_Y_COORDINATE', 'FEP_BUILDING_X_COORDINATE',  'FEP_BUILDING_DESC', 'FEP_BUILDING_CLASS']).size().reset_index(name='COUNT')
colors = {0: [0, 0, 255], 1: [0, 255, 0], 2: [255, 255, 0], 3: [255, 127, 0], 4: [255, 0, 0]}

if df_filter.size != 0:
  df_filter['COLOR'] = pd.cut(df_filter['COUNT'], bins=5, labels=False)
  df_filter['COLOR'] = df_filter['COLOR'].map(colors)

records = df_filter.to_dict('records')

layer = pdk.Layer(
  'ColumnLayer',
  data=records,
  diskResolution= 12,
  extruded=True,
  radius=10,
  elevationScale= 500 / df_filter['COUNT'].max(),
  get_position=['FEP_BUILDING_Y_COORDINATE', 'FEP_BUILDING_X_COORDINATE'],
  get_fill_color='COLOR',
  getElevation = 'COUNT',
  pickable=True
)

# Set the viewport location
view_state = pdk.ViewState(
    longitude=-95.404182,
    latitude=29.717154,
    zoom=14.7,
    min_zoom=14,
    max_zoom=20,
    pitch=45)

st.pydeck_chart(pdk.Deck(
    layers=[layer], 
    initial_view_state=view_state, 
    tooltip={"text": "Building: {FEP_BUILDING_DESC}\n Type: {FEP_BUILDING_CLASS}\n Tickets: {COUNT} "}
    ), height=700)